In [1]:
# import sys
# !{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install --upgrade python-dotenv
# !{sys.executable} -m pip install --upgrade watermark
# !{sys.executable} -m pip install --upgrade pydf
# !{sys.executable} -m pip install --upgrade google-genai

## Constants

In [2]:
# Directory to find Form 10K
FORM_10K_DIR = 'form_10k'

# Alphabet symbol
ALPHABET = 'GOOG'
# Alphabet Form 10K
ALPHABET_10K = '0ca59726-c36c-45fe-a114-e147f8cd7237.pdf'
# Tesla symbol
TSLA = 'TSLA'
# Tesla Form 10K
TSLA_10K = 'tsla-20241231-gen.pdf'

# Prefix for the Consolidated Balance Sheets
CBS = 'CBS'
# Prefix for Consolidated Statements of Operations
CSO = 'CSO'

# Model name
MODEL = 'gemini-2.0-flash-lite'

## Load API Keys

In [3]:
# To read environment property file
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()
# Check the environment variable
assert os.getenv('GEMINI_API_KEY') != None, 'Gemini API key is required'

## Create GenAI client

In [4]:
from google import genai

client = genai.Client()
assert client != None, 'Client instance is required'

## Copy pages to a new PDF
### Code from Goolgle AI with minor modifications

In [5]:
from pathlib import PosixPath
from pypdf import PdfReader, PdfWriter

def copy_pdf_pages(input_pdf_path:PosixPath, output_pdf_path:PosixPath, pages_to_copy:list[int]):
    """
    Copies specified pages from an input PDF to a new output PDF.

    Args:
        input_pdf_path (PosixPath): Path to the input PDF file.
        output_pdf_path (PosixPath): Path to the output PDF file.
        pages_to_copy (list): A list of 0-indexed page numbers to copy.
                              For example, [0, 2, 4] to copy the first, third, and fifth pages.
    """
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()

    for page_num in pages_to_copy:
        if 0 <= page_num < len(reader.pages):
            writer.add_page(reader.pages[page_num])
        else:
            print(f'Warning: Page {page_num + 1} is out of bounds for {input_pdf_path}. Skipping.')

    with open(output_pdf_path, 'wb') as output_file:
        writer.write(output_file)
    print(f'Successfully copied pages {pages_to_copy} from {input_pdf_path} to {output_pdf_path}.')

## Extract relevant pages

In [6]:
from pathlib import Path

# Construct the path to the Alphabet form 10K
f10k_path = Path(FORM_10K_DIR) / ALPHABET_10K
# Construct the path to the consolidated balance sheet for Alphabet
cbs_path = Path(FORM_10K_DIR) / f'{ALPHABET}_{CBS}.pdf'
# Copy page 52 (consolidated balance sheet)
copy_pdf_pages(f10k_path, cbs_path, [52])

# Construct the path to the Tesla form 10K
f10k_path = Path(FORM_10K_DIR) / TSLA_10K
# Construct the path to the consolidated statement of operations for Tesla
cso_path = Path(FORM_10K_DIR) / f'{TSLA}_{CSO}.pdf'
# Copy page 51 (consolidated statement of operations)
copy_pdf_pages(f10k_path, cso_path, [51])

Successfully copied pages [52] from form_10k/0ca59726-c36c-45fe-a114-e147f8cd7237.pdf to form_10k/GOOG_CBS.pdf.
Successfully copied pages [51] from form_10k/tsla-20241231-gen.pdf to form_10k/TSLA_CSO.pdf.


## Call Gemini

In [7]:
from google.genai import types
import pathlib

def call_gemini(doc_path:PosixPath, prompt:str):
    response = client.models.generate_content(
      model=MODEL,
      contents=[
          types.Part.from_bytes(
            data=doc_path.read_bytes(),
            mime_type='application/pdf',
          ),
          prompt])
    return response.text

## Query for financial periods - Alphabet

In [8]:
prompt = 'What financial periods are supported?'
print(call_gemini(doc_path=cbs_path, prompt=prompt))

The financial periods supported are:

*   **2023**
*   **2024**


## Query for Total Assets - Alphabet

In [9]:
prompt = 'Get the total assets for years 2023 and 2024'
print(call_gemini(doc_path=cbs_path, prompt=prompt))

Here's the information from the balance sheet:

*   **2023 Total Assets:** $402,392
*   **2024 Total Assets:** $450,256


## Query for Total Current Liabilities - Alphabet

In [10]:
prompt = 'Get the total current liabilities for years 2023 and 2024'
print(call_gemini(doc_path=cbs_path, prompt=prompt))

Here are the total current liabilities for 2023 and 2024, according to the provided image:

*   **2023:** $81,814 million
*   **2024:** $89,122 million


## Query for Goodwill change - Alphabet

In [11]:
prompt = 'Any significant changes to Goodwill between two periods?'
print(call_gemini(doc_path=cbs_path, prompt=prompt))

Here's an analysis of the Goodwill figures between 2023 and 2024:

*   **Goodwill - 2023:** $29,198 million
*   **Goodwill - 2024:** $31,885 million

**Change:** The Goodwill increased from $29,198 million to $31,885 million.
The difference between these two values is $2,687 million.


## Query for financial periods - Tesla

In [12]:
prompt = 'What financial periods are supported?'
print(call_gemini(doc_path=cso_path, prompt=prompt))

The financial periods supported are for the years ending December 31, 2024, 2023, and 2022.



## Query for Gross Profit - Tesla

In [13]:
prompt = 'What are the gross profits for 3 periods?'
print(call_gemini(doc_path=cso_path, prompt=prompt))

Here are the gross profits for the three periods:

*   **2024:** 17,450
*   **2023:** 17,660
*   **2022:** 20,853


## Query for Net Income - Tesla

In [14]:
prompt = 'How was the net income for 3 periods?'
print(call_gemini(doc_path=cso_path, prompt=prompt))

Here's the net income for the three periods, as reported in the image:

*   **2024:** $7,153 million
*   **2023:** $14,974 million
*   **2022:** $12,587 million


## Query for Research - Tesla

In [15]:
prompt = 'Has the research spending gone up for last 3 periods?'
print(call_gemini(doc_path=cso_path, prompt=prompt))

Let's analyze the research and development spending from the provided financial statements:

*   **2024:** $4,540 million
*   **2023:** $3,969 million
*   **2022:** $3,075 million

Looking at the amounts, the research and development spending has consistently **increased** over the last three periods (2022 to 2024).


## Display Modules

In [16]:
from watermark import watermark

%load_ext watermark
%watermark --iversions

watermark: 2.5.0
pypdf    : 5.9.0

